In [ ]:
from conceptacp.sae.SDLens import HookedStableDiffusionXLPipeline
from conceptacp.sae.SAE import SparseAutoencoder
from conceptacp.utils.feature_select import generate_sparse_maps
import torch
import os
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
from PIL import Image
from collections import defaultdict
from tqdm.notebook import tqdm
device = torch.device("cuda:1")

In [ ]:
dtype = torch.float32
pipe = HookedStableDiffusionXLPipeline.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=dtype,
    device_map="balanced",
    variant=("fp16" if dtype==torch.float16 else None)
)

In [ ]:
pipe.set_progress_bar_config(disable=True)
path_to_checkpoints = '../checkpoints/'

code_to_block = {
    "down.2.1": "unet.down_blocks.2.attentions.1",
    "mid.0": "unet.mid_block.attentions.0",
    "up.0.1": "unet.up_blocks.0.attentions.1",
    "up.0.0": "unet.up_blocks.0.attentions.0"
}

saes_dict = {}
means_dict = {}

for code, block in code_to_block.items():
    sae = SparseAutoencoder.load_from_disk(
        os.path.join(path_to_checkpoints, f"{block}_k10_hidden5120_auxk256_bs4096_lr0.0001", "final"),
    )
    means = torch.load(
        os.path.join(path_to_checkpoints, f"{block}_k10_hidden5120_auxk256_bs4096_lr0.0001", "final", "mean.pt"),
        weights_only=True
    )
    saes_dict[code] = sae.to(device, dtype=dtype)
    means_dict[code] = means.to(device, dtype=dtype)

In [ ]:
import json
with open("../data/train_prompts/retain.json", "r") as f:
    retain_prompts = json.load(f)

with open("../data/train_prompts/forgot.json", "r") as f:
    forgot_prompts = json.load(f)

unlearn_types=[
    "nudity",
    "style_vangogh",
    "object_church",
    "object_parachute"
]


In [ ]:
timesteps = 1
import pathlib
for unlearn_type in unlearn_types:
    print(unlearn_type)
    retain_maps = defaultdict(list)
    forgot_maps = defaultdict(list)
    for img_class in retain_prompts[unlearn_type]:
        for prompt in tqdm(retain_prompts[unlearn_type][img_class], desc=f"Processing {img_class} prompts"):
            output, cache = pipe.run_with_cache(
                prompt,
                positions_to_cache=list(code_to_block.values()),
                save_input=True,
                save_output=True,
                num_inference_steps=timesteps,
                guidance_scale=0.0,
                generator=torch.Generator(device="cpu").manual_seed(42)
            )
            for code in code_to_block:
                sparse_maps = generate_sparse_maps(code_to_block, cache, code, saes_dict, device)
                retain_maps[code].append(sparse_maps)
    for img_class in forgot_prompts[unlearn_type]:
        for prompt in tqdm(forgot_prompts[unlearn_type][img_class], desc=f"Processing {img_class} prompts"):
            output, cache = pipe.run_with_cache(
                prompt,
                positions_to_cache=list(code_to_block.values()),
                save_input=True,
                save_output=True,
                num_inference_steps=timesteps,
                guidance_scale=0.0,
                generator=torch.Generator(device="cpu").manual_seed(42)
            )
            for code in code_to_block:
                sparse_maps = generate_sparse_maps(code_to_block, cache, code, saes_dict, device)
                forgot_maps[code].append(sparse_maps)
    
    retain_maps = {k: torch.stack(v) for k, v in retain_maps.items()}
    forgot_maps = {k: torch.stack(v) for k, v in forgot_maps.items()}
    
    retain_maps[code].shape
    forgot_maps[code].shape

    def mu_score(sparse_maps):
        sparse_maps_flattened = sparse_maps.reshape(-1, sparse_maps.shape[-1])
        sparse_maps_mean = sparse_maps_flattened.mean(dim=0)
        return sparse_maps_mean
    
    def score(sparse_maps0, sparse_maps1):
        mu0 = mu_score(sparse_maps0)
        mu1 = mu_score(sparse_maps1)
        return mu0 / mu0.sum(dim=-1) - mu1 / mu1.sum(dim=-1)
    
    top_k = 10
    block_top_scores = {}
    for code in code_to_block:
        feature_score = score(forgot_maps[code], retain_maps[code])
        sorted_indices = torch.argsort(feature_score, descending=True)
        top_score = feature_score[sorted_indices[0]].item()
        block_top_scores[code] = top_score
        print(f"Top {top_k} feature indices for {code}: {sorted_indices[:top_k]}")
        print(f"Top {top_k} feature scores for {code}: {feature_score[sorted_indices[:top_k]]}")
        
    target_block = max(block_top_scores, key=block_top_scores.get)
    print(f"Selected block: {target_block}")

    feature_score = score(forgot_maps[target_block], retain_maps[target_block])
    sorted_indices = torch.argsort(feature_score, descending=True)

    max_score = feature_score[sorted_indices[0]].item()
    threshold = max_score / 5

    selected_features = []
    selected_strengths = []

    for idx in sorted_indices:
        score_value = feature_score[idx].item()
        if score_value >= threshold:
            if score_value == max_score:
                strength = -5.0
            else:
                strength = -5.0 * (score_value / max_score)
            strength = round(strength, 1)
            selected_features.append(idx.item())
            selected_strengths.append(strength)
        else:
            break

    result = {
        "unlearn_type": unlearn_type,
        "block": [target_block],
        "finetune_feature": selected_features,
        "finetune_strength": selected_strengths
    }

    output_path = pathlib.Path(f"/ConceptACP/data/finetune_config/{unlearn_type}.json")
    os.makedirs(output_path.parent, exist_ok=True)
    with open(output_path, 'a') as f:
        json.dump(result, f, indent=4)

    print(f"Config saved to {output_path}")